# Task #3.1 – Ensemble Methods

Goal: Enhance Decision Tree performance using ensemble methods, based on the optimized model from Task #3.0    
Deadline: 14th December

*Note:*
This task builds directly on Task #3.0; information like best-performing hyperparameters from the Decision Tree are reused to ensure a fair and focused comparison.

### Step 1 : Import all necessary classes

In [1]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Models
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier

# Preprocessing
from sklearn.preprocessing import LabelEncoder, label_binarize

# Metrics & validation
from sklearn.model_selection import StratifiedKFold, cross_val_predict, GridSearchCV
from sklearn.metrics import f1_score, recall_score, matthews_corrcoef, roc_auc_score

### Step 2 : Load the data from the csv file 

In [2]:
data = pd.read_csv('data/yeast.csv')
features = ['mcg', 'gvh', 'alm', 'mit', 'erl', 'pox', 'vac', 'nuc']
X = data[features].values
Y = data['name'].values
print(f"X shape : {X.shape}")
print(f"Y shape : {Y.shape}")

le = LabelEncoder()
Y_encoded = le.fit_transform(Y)

X shape : (1484, 8)
Y shape : (1484,)


### Step 3 : Create a baseline random forest

In [3]:
rf_base = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [4]:
Y_pred = cross_val_predict(rf_base, X, Y_encoded, cv=cv, method ='predict')
f1 = f1_score(Y_encoded, Y_pred, average='macro')
recall = recall_score(Y_encoded, Y_pred, average='macro')
mcc = matthews_corrcoef(Y_encoded, Y_pred)

Y_proba = cross_val_predict(rf_base, X, Y_encoded, cv=cv, method='predict_proba')
Y_bin = label_binarize(Y_encoded, classes=np.unique(Y_encoded))
auc_macro = roc_auc_score(Y_bin, Y_proba, average='macro', multi_class='ovr')

print("Baseline Random Forest Predictions : ")
print(f"F1 (macro): {f1:.4f}")
print(f"Recall (macro): {recall:.4f}")
print(f"MCC: {mcc:.4f}")
print(f"AUC (macro, OVR): {auc_macro:.4f}")

Baseline Random Forest Predictions : 
F1 (macro): 0.5774
Recall (macro): 0.5608
MCC: 0.5171
AUC (macro, OVR): 0.8754


#### Baseline Random Forest Predictions :      
F1 (macro): 0.5774      
Recall (macro): 0.5608     
MCC: 0.5171     
AUC (macro, OVR): 0.8754     

The baseline Random Forest significantly outperforms the single Decision Tree from the previous task. Bagging reduced variance and stabilized predictions, improving performance (class weighting helped mitigate class imbalance, leading to better macro-averaged metrics and a higher MCC)

### Step 4 : Compute best parameters for creating Random Forests and Best CV F1

In [5]:
rf_param_grid = {
    'n_estimators': [200, 400],
    'max_depth': [None, 15],
    'min_samples_leaf': [1, 5]
}
rf_grid = GridSearchCV(
    RandomForestClassifier(class_weight='balanced',random_state=42),
    rf_param_grid,
    scoring='f1_macro',
    cv=5,
    n_jobs=-1
)

rf_grid.fit(X, Y_encoded)

print(f"Best RF parameters: {rf_grid.best_params_}")
print(f"Best CV F1: {rf_grid.best_score_:.4f}")

Best RF parameters: {'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 400}
Best CV F1: 0.5632


Best RF parameters: {'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 400}       
Best CV F1: 0.5632             

The best cross-validation score reported by GridSearchCV is slightly lower than the baseline score obtained using cross_val_predict. 
        
This is expected because GridSearchCV computes metrics independently on each fold and then averages them, which penalizes poor performance on rare classes more strongly. In contrast, cross_val_predict aggregates all out-of-fold predictions before computing a single global metric, which can appear slightly more optimistic. 

### Step 5 : Defining baseline gradient boosting

In [6]:
gb_base = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)

In [7]:
Y_pred_gb = cross_val_predict(gb_base,X,Y_encoded,cv=cv,method='predict')
f1 = f1_score(Y_encoded, Y_pred_gb, average='macro')
recall = recall_score(Y_encoded, Y_pred_gb, average='macro')
mcc = matthews_corrcoef(Y_encoded, Y_pred_gb)

Y_proba_gb = cross_val_predict(gb_base,X,Y_encoded,cv=cv,method='predict_proba')
auc_macro = roc_auc_score(Y_bin, Y_proba_gb, average='macro', multi_class='ovr')

print("Baseline Gradient Boosting Predictions : ")
print(f"F1 (macro): {f1:.4f}")
print(f"Recall (macro): {recall:.4f}")
print(f"MCC: {mcc:.4f}")
print(f"AUC (macro, OVR): {auc_macro:.4f}")

Baseline Gradient Boosting Predictions : 
F1 (macro): 0.5341
Recall (macro): 0.5176
MCC: 0.4808
AUC (macro, OVR): 0.8473


#### Baseline Gradient Boosting Predictions :      
F1 (macro): 0.5341    
Recall (macro): 0.5176        
MCC: 0.4808            
AUC (macro, OVR): 0.8473           

Baseline Gradient Boosting performs better than a single Decision Tree but slightly worse than Random Forest. While boosting reduces bias by sequentially correcting errors, it is more sensitive to class imbalance and noisy minority classes, which limits its overall macro-averaged performance on this dataset.


### Step 6 : Tuning Gradient Boosting to compute best GB parameters

In [8]:
gb_param_grid = {
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1],
    'max_depth': [2, 3],
    'subsample': [0.8, 1.0]
}

In [9]:
gb_grid = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    gb_param_grid,
    scoring='f1_macro',
    cv=5,
    n_jobs=-1
)

gb_grid.fit(X, Y_encoded)

print(f"Best GB parameters: {gb_grid.best_params_}")
print(f"Best CV F1: {gb_grid.best_score_:.4f}")

Best GB parameters: {'learning_rate': 0.05, 'max_depth': 2, 'n_estimators': 100, 'subsample': 0.8}
Best CV F1: 0.5561


Best GB parameters: {'learning_rate': 0.05, 'max_depth': 2, 'n_estimators': 100, 'subsample': 0.8}      
Best CV F1: 0.5561            

Hyperparameter tuning improves Gradient Boosting performance by reducing overfitting and improving generalization. A lower learning rate, shallow trees, and subsampling result in smoother decision boundaries and better handling of minority classes, leading to an improved cross-validated F1 score compared to the baseline model.


### Additional Step :       
Since XGBoost does not work in my python environment, I have used Hist Gradient Boosting as a comparable alternative to measure performance against Random Forests and Gradient Boosting

In [10]:
hgb_base = HistGradientBoostingClassifier(
    max_depth=6,
    learning_rate=0.1,
    random_state=42
)

In [11]:
Y_pred_hgb = cross_val_predict(hgb_base,X,Y_encoded,cv=cv,method='predict')
f1 = f1_score(Y_encoded, Y_pred_hgb, average='macro')
recall = recall_score(Y_encoded, Y_pred_hgb, average='macro')
mcc = matthews_corrcoef(Y_encoded, Y_pred_hgb)

Y_proba_hgb = cross_val_predict(hgb_base,X,Y_encoded,cv=cv,method='predict_proba')
auc_macro = roc_auc_score(Y_bin, Y_proba_hgb, average='macro', multi_class='ovr')

print("Baseline HistGradient Boosting Predictions : ")
print(f"F1 (macro): {f1:.4f}")
print(f"Recall (macro): {recall:.4f}")
print(f"MCC: {mcc:.4f}")
print(f"AUC (macro, OVR): {auc_macro:.4f}")

Baseline HistGradient Boosting Predictions : 
F1 (macro): 0.4308
Recall (macro): 0.4227
MCC: 0.4596
AUC (macro, OVR): 0.8246


#### Baseline HistGradient Boosting Predictions :          
F1 (macro): 0.4308        
Recall (macro): 0.4227        
MCC: 0.4596       
AUC (macro, OVR): 0.8246       

Although HistGradientBoostingClassifier is designed for efficiency on large datasets, it performed worse than standard Gradient Boosting on this task. This is attributed to the small dataset size, limited feature dimensionality, and inability to directly handle class imbalance, making histogram-based optimization ineffective in this context.

# Results :
| Model                | F1 Macro  | MCC       | AUC       | Verdict      |
| -------------------- | --------- | --------- | --------- | ------------ |
| Decision Tree        | ~0.47     | ~0.38     | ~0.78     | Baseline     |
| Random Forest        | **~0.58** | **~0.52** | **~0.88** | Best         |
| Gradient Boosting    | ~0.55     | ~0.48     | ~0.85     | Competitive  |
| HistGradientBoosting | ~0.43     | ~0.46     | ~0.82     | Not suitable |


### Overall Comparison

Among all models, Random Forest achieves the best overall performance due to effective variance reduction through bagging. Gradient Boosting benefits from tuning but remains slightly less robust to class imbalance. AdaBoost performs poorly due to sensitivity to noisy and extremely small classes. Overall, ensemble methods substantially outperform the single Decision Tree, confirming the effectiveness of both bagging and boosting for this task.